Instead of maintaining two separate classes, MultiHeadAttentionWrapper and CausalAttention, we can combine both these into one single MultiHeadAttention class

Also in addition to this, we will add some more modifications to implement multi head attention more efficiently

In MultiHeadAttentionWrapper, multiple heads are implemented by stacking a list of CausalAttention objects (self.heads), each representing a separate attention head

The CausalAttention class independently computes the context vectors and concatenates the result from each head

In contrast, this method integrates the multi head functionality in a single class

It splits the input into multiple heads by reshaping the projected q,k,v tensors and then combines the results from these heads after computing attention

In [1]:
import torch

In [10]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False): #d_in is the input embedding size, and d_out is the output embedding size after attention—they can be the same or different depending on the design.
        super().__init__()
        assert (d_out%num_heads==0), "Output dimension must be divisible by number of heads"
        self.d_out = d_out 
        self.num_heads = num_heads
        self.head_dim = d_out//num_heads # each head dimension will have the same number of features. we define the head dim here
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = torch.nn.Linear(d_out, d_out) #Linear layer to combine the different head outputs
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer(
    "mask",
    torch.triu(torch.ones(context_length, context_length), diagonal=1)
)


    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys=self.W_key(x) #shape = b, num_tokens, d_out
        queries = self.W_query(x)
        values = self.W_value(x)

        # We split the matrix by adding a num_heads dimension
        # unroll the last dimension : (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        #Transpose
        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # Compute scaled dot product attention (self attention) using causal mask
        attn_scores = queries @ keys.transpose(2,3) # dot product for each head

         # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)   

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

Step 1: Reduce the projection dim to match desired output dim

Step 2: Use a Linear layer to combine head outputs

Step 3: Tensor shape: (b, num_tokens, d_out)

Step 4: We implicitly split the matrix by adding a num_heads dimension. Then we unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)

Step 5: Transpose from shape (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)

Step 6: Compute dot product for each head

Step 7: Mask truncated to the number of tokens

Step 8: Use the mask to fill attention scores

Step 9: Tensor shape: (b, num_tokens, n_heads, head_dim)

Step 10: Combine heads, where self.d_out = self.num_heads * self.head_dim

Step 11: Add an optional linear projection

### why do we split the output dimension into num_heads parts?
-  instead of doing one big attention, we do multiple smaller attentions in parallel
- this is so that multiple attention heads can learn different things at the same time
- to do that, we split the features into chunks where each chunk gets one head
- each head must get the same number of features
- d_out ÷ num_heads = features per head
- Then you combine them back together at the end


#### d_in and d_out difference
d_in is the input embedding size, and d_out is the output embedding size after attention—they can be the same or different depending on the design.

#### context_length 
maximum number of tokens the model can look at at once

#### Why do we reshape here? i.e. apply view

keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)

- Originally the shape is: (b, num_tokens, d_out)

- The last dimension is one long vector.

- After view, you split that last dimension into two dimensions:
- d_out -> (num_heads, head_dim)

- So:(b, num_tokens, d_out) becomes: (b, num_tokens, num_heads, head_dim)
- this means: Take each token’s long vector and split it into smaller chunks, one chunk per attention head


In [7]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 

torch.Size([2, 3, 6])


In [13]:
batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.1093, -0.3787,  0.2819, -0.0415, -0.2864, -0.0641],
         [ 0.1579, -0.3743,  0.2606, -0.0959, -0.2973,  0.0021],
         [ 0.1650, -0.3640,  0.2507, -0.1197, -0.2805,  0.0862]],

        [[ 0.1093, -0.3787,  0.2819, -0.0415, -0.2864, -0.0641],
         [ 0.1579, -0.3743,  0.2606, -0.0959, -0.2973,  0.0021],
         [ 0.1650, -0.3640,  0.2507, -0.1197, -0.2805,  0.0862]]],
       grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 3, 6])


While the code is fully functional, we used relatively small embedding sizes and numbers of attention heads to keep the outputs readable.

For comparison, the smallest GPT-2 model (117 million parameters) has 12 attention heads and a context vector embedding size of 768.

The largest GPT-2 model (1.5 billion parameters) has 25 attention heads and a context vector embedding size of 1600.

Note that the embedding sizes of the token inputs and context embeddings are the same in GPT models (d_in = d_out).